In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("..")

In [3]:
from IPython.display import clear_output
from src.dataset_loaders_new import get_samplers, get_indices
from src.utils import get_pca_models
from src import utils
from src.train import train_discrete
import wandb
from torch.utils.data import TensorDataset, DataLoader
import yaml
import numpy as np
import random
import pickle
import gc

## 1. Parameters.

Possible ```DATASET_NAME``` values are: ```twitter```, ```wiki-gigaword```, ```bone_marrow```

SOURCE_DIM   = 100
TARGET_DIM   = 50
MAX_ITERS    = 100

In [11]:

DATASET_NAME = 'twitter'
METHOD_NAME  = 'RegGW'
DEVICE       = 'cpu'

ALPHA        = 0.5
SEED         = 43
COST_DISCRETE= 'cosine'

config = {'dataset':dict(DATASET_NAME         = DATASET_NAME,
                         DEVICE               = DEVICE,
                         SOURCE_DIM           = SOURCE_DIM,
                         TARGET_DIM           = TARGET_DIM,
                         FUSED_DIM            = 0,
                         MAX_WORDS            = 7024,#400000,
                         N_SAMPLES_DISCRETE   = 6000,
                         N_SAMPLES_CONTINUOUS = 256,
                         N_EVAL               = 4,
                         ALPHA                = ALPHA, 
                         BATCH_SIZE_TRAIN     = 256,
                         BATCH_SIZE_TEST      = 256,
                         SEED                 = SEED,
                         TRAIN_TYPE           = 'discrete',
                         NORMALIZE_VECS       = False
                          ),
          
          'training':dict(METHOD_NAME          = METHOD_NAME,
                          MAX_ITERS            = MAX_ITERS,
                          COST_DISCRETE        = COST_DISCRETE,
                          ),

          'model_specific':dict(HIDDEN_SIZES_MLP= [512, 256, 256],
                                EPSILON        = 1e-3                  
                               )}



## 2. Loading dataset.

In [12]:
dataset_path = '../datasets'
sys.path.append(dataset_path)

source_vectors, target_vectors, random_indices_train, random_indices_test = get_indices(dataset_path, config)

print(source_vectors.shape)
print(target_vectors.shape)
print(random_indices_train.shape)

KeyError: 'N_EVAL'

## 3. Training.

In [13]:
import os
#os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'

n_repeats = 1
wandb_report = False
project_name = f'{METHOD_NAME}_{DATASET_NAME}_{SOURCE_DIM}->{TARGET_DIM}_6K_5rep'

metrics_names = ['Top@1', 'Top@5', 'Top@10', 'cossim_gt', 'inner_gw', 'foscttm']
_, _, _, random_indices_test_fixed = get_indices(dataset_path, config)      

alpha_values = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0][::-1]
#alpha_values = [0.0][::-1]

metrics_out = {str(np.round(alpha, 1)):[] for alpha in alpha_values}

for ALPHA in alpha_values:
    
    config['dataset']['ALPHA'] = ALPHA 
    
    print('================================')
    print(f'Experiment for ALPHA={ALPHA}')
    print('================================')
    
    for ix in range(n_repeats):
        
        if wandb_report:
            exp_name = f'ALPHA_{np.round(ALPHA, 1)}_repeat_{ix}'
            wandb.init(name=exp_name, config=config, project=project_name)
            
        SEED = random.randint(0, 10000)
        config['dataset']['SEED'] = SEED
        print('Seed: ', SEED)
        
        source_vectors, target_vectors, random_indices_train, _ = get_indices(dataset_path, config)            
        source_vectors, target_vectors, train_sampler, test_sampler = get_samplers(source_vectors, target_vectors, random_indices_train, random_indices_test_fixed, config)

        #pca_models = get_pca_models(source_vectors, target_vectors)
        
        trained_class, metrics_dict = train_discrete(train_sampler, test_sampler, 
                                                     metrics_names, target_vectors,
                                                     config,
                                                     wandb_report=wandb_report,
                                                     axis_lims=None, report_every=1000)
        
        metrics_out[str(np.round(ALPHA, 1))].append(metrics_dict)

        with open(f'results_discrete/{project_name}.pkl', 'wb') as f:
            pickle.dump(metrics_out, f)

        del trained_class, metrics_dict, source_vectors, target_vectors, random_indices_train, train_sampler, test_sampler
        gc.collect()

KeyError: 'N_EVAL'

In [8]:
with open(f'results_discrete/{project_name}.pkl', 'wb') as f:
    pickle.dump(metrics_out, f)

In [9]:
print(metrics_out)

{'0.3': [{'train': [{'Top@1': {'mean': 0.137, 'std': 0.0}, 'Top@5': {'mean': 0.30433333333333334, 'std': 0.0}, 'Top@10': {'mean': 0.409, 'std': 0.0}, 'cossim_gt': {'mean': 0.1845218539237976, 'std': 0.0}, 'inner_gw': {'mean': 74.62925720214844, 'std': 0.0}, 'foscttm': {'mean': 0.494, 'std': 0.0}}], 'test': [{'Top@1': {'mean': 0.111328125, 'std': 0.011554843326366438}, 'Top@5': {'mean': 0.2802734375, 'std': 0.021103694125951474}, 'Top@10': {'mean': 0.38671875, 'std': 0.03149319432929121}, 'cossim_gt': {'mean': 0.734306737780571, 'std': 0.0038035706087871635}, 'inner_gw': {'mean': 75.98898696899414, 'std': 4.53276888894101}, 'foscttm': {'mean': 0.132525, 'std': 0.002170685375636002}}]}, {'train': [{'Top@1': {'mean': 0.14533333333333334, 'std': 0.0}, 'Top@5': {'mean': 0.31566666666666665, 'std': 0.0}, 'Top@10': {'mean': 0.4156666666666667, 'std': 0.0}, 'cossim_gt': {'mean': 0.17035965621471405, 'std': 0.0}, 'inner_gw': {'mean': 72.71646118164062, 'std': 0.0}, 'foscttm': {'mean': 0.5021, '

In [10]:
name = 'CONT_twitter_100->50_AlignGW_seed_43.pkl'

with open(f'results_discrete/{name}', 'rb') as f:
    x = pickle.load(f)

FileNotFoundError: [Errno 2] No such file or directory: 'results_discrete/CONT_twitter_100->50_AlignGW_seed_43.pkl'

In [ ]:
print(x)